# Stage 4 — RE-SCREEN the top blades (no re-optimization)

Re-evaluates the **already-carved** density fields a prior Stage-4 TO run wrote to Drive
(`<name>_density.npy`) — **one factorization per design, not the ~160 of a full TO**. Use it to
get the *honest* structural numbers without re-running the optimization:

- **carved-mass inertial deflection** (~7-8 mm, the as-printed value — not the conservative full-solid ~10 mm),
- **solid-only von Mises** — peak stress over only the elements that survive to the print (`rho >= 0.5`),
  so you can tell a real overstress from a gray-element artifact (the design-00 question).

Prereq: the Stage-4 TO run already produced `<OUT_DIR>/summary.json` + the `_density.npy` fields.
Mesh size / skin are read back from that `summary.json`, so they can't drift.

## 1. Connect Drive

In [ ]:
# Drive connect ONLY (kept separate from the repo/deps install below).
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = Path.cwd() / "data"
print("drive root:", DRIVE_ROOT)

## 2. Repo + deps  (same stack as the TO notebook)

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the TO tool + this notebook land on main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    # TO stack: gmsh + CadQuery for the solid mesh, scikit-fem for the 3D FEA.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gmsh", "cadquery", "scikit-fem"], check=True)
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO)

## 3. Config — point at the Stage-4 TO outputs

In [ ]:
from fanopt.geometry.schema import SIGMA_Y_PETG_Z_PA

# This screens the CARVED (TO) density fields in OUT_DIR - NOT the raw BO designs. SHARED_DIR +
# VERIFICATION are used ONLY to (a) pick which 4 designs by J_fan rank and (b) fetch each design's
# geometry params to REBUILD its mesh (a saved _density.npy carries no mesh). The thing evaluated is
# the TO-carved field laid onto that mesh: BO gives the shape, TO carves it, this screens the carving.
SHARED_DIR   = DRIVE_ROOT / "campaign_trapezoid"                       # campaign shards (to rebuild the mesh)
VERIFICATION = DRIVE_ROOT / "stage3_verify_blade" / "verification.json"  # Stage-3.C fine J_fan (picks the top-4)
OUT_DIR      = DRIVE_ROOT / "stage4_blade_to"                          # <-- the TO carved fields SCREENED here

TOP_K      = 4     # the 4 best designs by FINE J_fan (wind). NOTE: the NN_ name prefix is the
                   # COARSE rank, so the fine-best four are NOT necessarily names 00-03 - selection
                   # is by fine J_fan and the results table below sorts by it.
STRESS_FOS = 2.0   # allowable = PETG weak-axis yield / FOS

# Re-screen is ONE factorization per design (no OC loop) and skips the density filter, so each worker
# is LIGHTER than a TO worker (~8-12 GB at 0.6mm). 4 workers run the 4 designs at once.
N_WORKERS  = 4

SIGMA_ALLOW_MPA = SIGMA_Y_PETG_Z_PA / STRESS_FOS / 1e6
assert OUT_DIR.exists(), f"Stage-4 output folder not found: {OUT_DIR}"
assert (OUT_DIR / "summary.json").exists(), f"no summary.json in {OUT_DIR} - run the TO first"
print(f"re-screen top-{TOP_K} from {OUT_DIR}")
print(f"stress allowable = {SIGMA_ALLOW_MPA:.1f} MPa (yield {SIGMA_Y_PETG_Z_PA/1e6:.0f} / FOS {STRESS_FOS:g}) | {N_WORKERS} workers")


## 4. RUN the re-screen

Loads each **TO-carved** density field from `OUT_DIR`, rebuilds the FEA model, and screens it once.
(The verification file only picks the top-4 and supplies geometry to rebuild the mesh - the density
being screened is the TO output, not the raw BO design.) Mesh size / skin are read from
`OUT_DIR/summary.json` so the rebuilt mesh matches the saved density. ~5-10 min for 4 designs in
parallel. Each line prints `VM ... (solid-only ...)` + the per-load deflection breakdown.

In [ ]:
import rescreen_blade_to

summary = rescreen_blade_to.run(
    shared_dir=SHARED_DIR,
    verification=VERIFICATION,
    out_dir=OUT_DIR,
    top_k=TOP_K,
    n_workers=N_WORKERS,
    mesh_size_m=None,        # read from summary.json
    skin_thickness_m=None,   # read from summary.json
    stress_fos=STRESS_FOS,
    progress=True,
)
print(f"\nre-screened {summary['n_succeeded']}/{summary['n_designs']} -> {OUT_DIR}/rescreen_summary.json")


## 5. Results — honest deflection + all-vs-solid stress

The key column is **solid-only VM**: if it is well under the allowable while the all-element VM is not,
that design's stress "failure" was a gray-element artifact and it is actually printable. Deflection is
shown for context but the ~1 mm screen limit is advisory (ADR-0007/0008) — the real cert is the feel test.

In [ ]:
import json

resc = json.loads((OUT_DIR / "rescreen_summary.json").read_text())
rows = [r for r in resc["designs"] if "error" not in r]
# Sort by FINE J_fan (wind) descending - the real ranking. The NN_ name prefix is only coarse rank.
rows.sort(key=lambda r: r.get("j_fan_3d") or 0.0, reverse=True)

print(f"{'name':26} {'J_fan_3d':>10} {'mass_g':>7} {'rem%':>6} {'u_tip_mm':>9} {'VM_all':>7} {'VM_solid':>9} {'stress':>8}")
for r in rows:
    vm_solid = r.get("max_von_mises_solid_mpa")
    stress_ok = "PASS" if (vm_solid is not None and vm_solid <= SIGMA_ALLOW_MPA) else "FAIL"
    j3d = r.get("j_fan_3d")
    print(f"{r['name']:26} {(('%.3e' % j3d) if j3d else '    -    '):>10} {r['mass_kg']*1e3:7.1f} "
          f"{r['volume_removed_frac']*100:6.1f} {r['u_tip_max_mm']:9.3f} {r['max_von_mises_mpa']:7.2f} "
          f"{(vm_solid if vm_solid is not None else float('nan')):9.2f} {stress_ok:>8}")
for r in resc["designs"]:
    if "error" in r:
        print(f"  [error] {r['name']}: {r['error']}")
if any(r.get("remap_max_dist_m") for r in rows):
    print("\n[!] non-zero remap drift on some designs - the re-screen rebuilt a slightly different mesh")
    print("    than the TO run. Skin is re-pinned so this is usually fine; large drift is hard-failed.")

print(f"\nstress allowable = {SIGMA_ALLOW_MPA:.1f} MPa.  VM_all counts gray (intermediate-density) elements")
print("and OVERSTATES true stress; VM_solid counts only rho>=0.5 (as-printed) elements = the honest peak.")
print("If VM_solid passes while VM_all fails, the design's stress 'failure' was a gray-element artifact.")

# Per-load deflection breakdown (which load drives the flex — inertial is the wrist-snap turning point).
print("\nper-load tip deflection (mm):")
for r in rows:
    by_load = "  ".join(f"{k} {v:.2f}" for k, v in r.get("u_tip_by_load_mm", {}).items())
    print(f"  {r['name']}: {by_load}")


## 6. Render the 4 carved blades (retained material after TO)

3D point cloud of the material each design KEEPS (density > 0.5) — the carved internal rib/interior
structure TO produced (the frozen aero skin shows as the dense outer shell). Rotate/zoom each panel.
This is the *carved* field, not the folded fan assembly — that render is separate.

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

resc = json.loads((OUT_DIR / "rescreen_summary.json").read_text())
shown = [r for r in resc["designs"] if "error" not in r]
shown.sort(key=lambda r: r.get("j_fan_3d") or 0.0, reverse=True)  # best wind first
n = len(shown)
fig = make_subplots(rows=1, cols=n, specs=[[{"type": "scene"}] * n],
                    subplot_titles=[f"{r['name'][:11]}  J={r.get('j_fan_3d') or 0:.2e}" for r in shown])
for col, r in enumerate(shown, start=1):
    name = r["name"]
    dens = np.load(OUT_DIR / f"{name}_density.npy")
    cen = np.load(OUT_DIR / f"{name}_centroids.npy")   # saved by the TO batch; aligned to dens
    keep = dens > 0.5                                    # material retained after TO
    fig.add_trace(go.Scatter3d(
        x=cen[keep, 0], y=cen[keep, 1], z=cen[keep, 2], mode="markers",
        marker=dict(size=1.2, color=dens[keep], colorscale="Viridis", cmin=0.5, cmax=1.0),
        showlegend=False), row=1, col=col)
fig.update_layout(height=460, width=300 * n,
                  title="Retained material after TO (density > 0.5) — sorted by J_fan (best wind first)")
fig.show()
